# Importar pacotes

In [60]:
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# Métodos utilitários

In [61]:
from utils.common import import_dataframe
from utils.common import make_lags, make_leads
from utils.common import calculate_metrics
from utils.common import HybridRecursive, HybridBase
from utils.common import ModeloPrevisaoVolume

# Importação dos dados

In [62]:
nome_coluna_volume = "Volume Útil Armazenado (%)"
nome_coluna_vazao_natural = "Vazão Natural (m³/s)"
nome_coluna_vazao_jusante = "Vazão Jusante (m³/s)"

df = import_dataframe()
df.set_index("Data", inplace=True)
df = df[df.index >= "2018-01-01"]

#df_volume = df[["Data", nome_coluna_volume]].copy()
#df_vazao_natural = df[["Data", nome_coluna_vazao_natural]].copy()
#df_vazao_jusante = df[["Data", nome_coluna_vazao_jusante]].copy()

#df_volume = df_volume.set_index("Data")
#df_vazao_natural = df_vazao_natural.set_index("Data")
#df_vazao_jusante = df_vazao_jusante.set_index("Data")

#df_volume_series = (
#  df_volume
#    .groupby('Data').mean()
#    .squeeze()
#)
#df_vazao_natural_series = (
#  df_vazao_natural
#    .groupby('Data').mean()
#    .squeeze()
#)

#df_vazao_jusante_series = (
#  df_vazao_jusante
#    .groupby('Data').mean()
#    .squeeze()
#)

# Construção - Vazão Natural  e Juzante

In [63]:

VALIDATION_SIZE = 1*90

def vazao_dados(df, nome_coluna_vazao):
  y_vazao = df[nome_coluna_vazao]
  fourier = CalendarFourier(freq="YE", order=2)
  y_vazao = y_vazao.asfreq("D")
  dp = DeterministicProcess(
    index=y_vazao.index,
    constant=False,
    order=1,
    seasonal=False,
    additional_terms=[fourier],
    drop=True,
  )
  
  X_full_vazao = dp.in_sample()
  return X_full_vazao, y_vazao

X_j, y_j = vazao_dados(df, nome_coluna_vazao_jusante)
X_n, y_n = vazao_dados(df, nome_coluna_vazao_natural)
df_aux = pd.concat([pd.DataFrame(X_j).add_suffix('_jusante'),
                    pd.DataFrame(X_n).add_suffix('_natural'),
                    y_n, y_j], axis=1)


In [64]:
#### DENTRO DO FOR PORQUE FAZ AUTOREGRESSÃO -> SO VALE PARA TREINO. 
def volume_dados(df, y_vn, y_vj, TARGET_COLUMN_NAME = 'Volume Útil Armazenado (%)'):
  y_vol = df[TARGET_COLUMN_NAME]
  X_lags = make_lags(y_vol.squeeze(), 1)
  X_Qin_leads = make_leads(y_vn.squeeze(), 1, name="Qin")
  X_Qout_leads = make_leads(y_vj.squeeze(), 1, name="Qout")
  X_full_vol = pd.concat([X_lags, X_Qin_leads, X_Qout_leads], axis=1).dropna()
  y_vol, X_full_vol = y_vol.align(X_full_vol, join='inner', axis=0)
  return X_full_vol, y_vol


## Treino

In [ ]:
N_SPLITS = 5
FORECAST_HORIZON = VALIDATION_SIZE
tscv = TimeSeriesSplit(
    n_splits=N_SPLITS,
    test_size=FORECAST_HORIZON,
    gap=0 # Sem gap entre treino e teste
)

all_preds_model = []
all_y_test = []
lista_modelos_v = [LinearRegression(fit_intercept=False)]
lista_modelos_j = [HybridRecursive(LinearRegression(), RandomForestRegressor(), lags=2) for i in range(0, len(lista_modelos_v))]
lista_modelos_n = [HybridRecursive(Lasso(), KNeighborsRegressor(), lags=2) for i in range(0, len(lista_modelos_v))]



#### FOR PARA CADA MODELO A SER TESTADO ###
for pos, modelo_v in enumerate(lista_modelos_v):

    model = ModeloPrevisaoVolume(modelo_v, usar_jusante=True,
                                modelo_vazao_juzante = lista_modelos_j[pos],
                                modelo_vazao_natural = lista_modelos_n[pos])


#### DENTRO DE UM FOR PARA CADA SPLIT DO TIME SERIES SPLIT ####
    for fold, (train_idx, test_idx) in enumerate(tscv.split(df)):
    
        df_train = df.iloc[train_idx]
        df_test = df.iloc[test_idx]
        df_aux_train = df_aux.iloc[train_idx]
        df_aux_test = df_aux.iloc[test_idx]        
        X_full_vol_train, y_vol_train = volume_dados(df_train, y_n, y_j)

        model.fit_vazao_jusante(df_aux_train.filter(like='jusante'), y_j.loc[df_aux_train.index])
        model.calculate_lags(df_aux_test.filter(like='jusante'))
        y_pred_train_vj = model.predict(df_aux_train.filter(like='jusante'))
        y_pred_test_vj = model.predict(df_aux_test.filter(like='jusante'))

        model.fit_vazao_natural(df_aux_test.filter(like='natural'), y_n.loc[df_aux_train.index])
        model.calculate_lags(df_aux_test.filter(like='natural'))
        y_pred_train_vn = model.predict(df_aux_train.filter(like='natural'))
        y_pred_test_vn = model.predict(df_aux_test.filter(like='natural'))
        
        model.fit(X_full_vol_train, y_vol_train)

        #### VAI FAZER O CALCULO E SALVAR A METRICA ####
        
        
        

PARA DESACOPLAR

Usar modelos separados.

1) Criar função Calculate lags independente de modelo ou de qualquer coisa (SEM CLASSE).
2) Fazer para cada tipo (VAZAO PRIMEIRO) o fit do modelo 1 (que considera sazonalidade e tendencia)
3) Fazer o predict deste modelo
4) Fazer yreal - y_pred (para calcular o ciclo Lag2), pegar o modelo 2 e fazer fit dele considerando lag do residuo.
5) Obter o valor predito final y_pred_final -> somando y_pred + y_pred_residuo
6) Pegar o y_pred_final (da vazao) e unir com o y_pred_final da outra vazao e usar o lag do volume como variáveis do modelo de volume que faz fit e predict.
OBS: O MODELO É RECURSIVO